[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/apartsin/DLCourseHIT/blob/main/notebooks/week12.ipynb)

# Week 12: Representation Learning
**Introduction to Deep Learning (HIT)** &middot; Part III: Architectures & Representation Learning

Autoencoders and latent representations; contrastive and self-supervised methods.

**Instructor practice notebook** for the 2-hour practice lesson. Work through the sections below live, running each cell and trying the variations. The student homework is the weekly lab.

### Goals

- Train an autoencoder and a contrastive embedding.
- Probe and interpret a learned latent space.
- Reason about what makes a representation useful.

### Setup
Run this first. On Colab, set Runtime > Change runtime type > GPU for the later weeks.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(0)
print('PyTorch', torch.__version__, '| device:', device)

## 1. Train an autoencoder on MNIST
Compress to a small bottleneck, then reconstruct (downloads ~10 MB).

In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
train = datasets.MNIST("./data", train=True, download=True, transform=transforms.ToTensor())
dl = DataLoader(train, batch_size=256, shuffle=True)

class AE(nn.Module):
    def __init__(self, latent=16):
        super().__init__()
        self.enc = nn.Sequential(nn.Flatten(), nn.Linear(784, 64), nn.ReLU(), nn.Linear(64, latent))
        self.dec = nn.Sequential(nn.Linear(latent, 64), nn.ReLU(), nn.Linear(64, 784), nn.Sigmoid())
    def forward(self, x):
        z = self.enc(x); return self.dec(z).view_as(x), z

ae = AE().to(device); opt = torch.optim.Adam(ae.parameters(), 1e-3); f = nn.MSELoss()
for epoch in range(2):
    for xb, _ in dl:
        xb = xb.to(device); opt.zero_grad(); rec, _ = ae(xb); f(rec, xb).backward(); opt.step()
    print(f"epoch {epoch}: reconstruction MSE {f(ae(xb)[0], xb).item():.4f}")

## 2. Reconstructions
Top row: originals; bottom row: reconstruction through a 16-d bottleneck.

In [ ]:
xb, yb = next(iter(dl)); xb = xb.to(device)
rec, _ = ae(xb); rec = rec.detach().cpu()
fig, ax = plt.subplots(2, 8, figsize=(12, 3))
for i in range(8):
    ax[0, i].imshow(xb[i, 0].cpu(), cmap="gray"); ax[0, i].axis("off")
    ax[1, i].imshow(rec[i, 0], cmap="gray"); ax[1, i].axis("off")
plt.show()

## 3. The bottleneck width matters
Wider latent = lower reconstruction error; the bottleneck forces compression.

In [ ]:
for latent in [2, 8, 32]:
    m = AE(latent).to(device); o = torch.optim.Adam(m.parameters(), 1e-3); f = nn.MSELoss()
    for xb, _ in dl:
        xb = xb.to(device); o.zero_grad(); rec, _ = m(xb); f(rec, xb).backward(); o.step()
    print(f"latent={latent:2d}: reconstruction MSE {f(m(xb)[0], xb).item():.4f}")

## 4. The latent space carries class structure
Encode the batch, project the 16-d code to 2-D with PCA, color by digit.

In [ ]:
with torch.no_grad():
    _, z = ae(xb)
z = z.cpu() - z.cpu().mean(0)
U, S, V = torch.pca_lowrank(z, q=2)
proj = z @ V[:, :2]
plt.scatter(proj[:, 0], proj[:, 1], c=yb[:len(proj)], cmap="tab10", s=10)
plt.title("Autoencoder latent space (PCA to 2-D)"); plt.colorbar(); plt.show()

## 5. A linear probe measures representation quality
Freeze the encoder, train a linear classifier on the latent code.

In [ ]:
with torch.no_grad():
    feats = ae.enc(xb).cpu()                  # frozen features
probe = nn.Linear(feats.shape[1], 10); o = torch.optim.Adam(probe.parameters(), 0.05); f = nn.CrossEntropyLoss()
for _ in range(200):
    o.zero_grad(); f(probe(feats), yb[:len(feats)]).backward(); o.step()
acc = (probe(feats).argmax(1) == yb[:len(feats)]).float().mean().item()
print(f"linear probe accuracy on frozen 16-d features: {acc:.3f}")

## 6. A denoising autoencoder
Corrupt the input, reconstruct the clean image: the model learns robust structure.

In [ ]:
noisy = (xb + 0.5 * torch.randn_like(xb)).clamp(0, 1)
dae = AE().to(device); o = torch.optim.Adam(dae.parameters(), 1e-3); f = nn.MSELoss()
for _ in range(200):
    o.zero_grad(); rec, _ = dae(noisy); f(rec, xb).backward(); o.step()   # input noisy, target clean
rec = dae(noisy)[0].detach().cpu()
fig, ax = plt.subplots(2, 6, figsize=(9, 3))
for i in range(6):
    ax[0, i].imshow(noisy[i, 0].cpu(), cmap="gray"); ax[0, i].set_title("noisy"); ax[0, i].axis("off")
    ax[1, i].imshow(rec[i, 0], cmap="gray"); ax[1, i].set_title("denoised"); ax[1, i].axis("off")
plt.show()

## 7. The contrastive idea
Two augmented views of one image form a positive pair; everything else is negative.

In [ ]:
from torchvision import transforms
aug = transforms.Compose([transforms.RandomResizedCrop(28, scale=(0.6, 1.0)), transforms.RandomHorizontalFlip()])
img = xb[0].cpu()
fig, ax = plt.subplots(1, 3, figsize=(6, 2))
for a_, t, title in zip(ax, [img, aug(img), aug(img)], ["original", "view 1", "view 2"]):
    a_.imshow(t[0], cmap="gray"); a_.set_title(title); a_.axis("off")
plt.show()

**Mini-exercise:** compare the linear-probe accuracy from a 2-d vs a 32-d bottleneck. Wider codes carry more class structure. **Recap:** a bottleneck forces useful compression; the latent code is the point, not perfect reconstruction; augmentation choices define what 'similar' means; learned features transfer.

---
Student materials for this week: the lab handout (`labs/week12.html`) and the curated references (`references/week12.html`) in the course site.